GEETA 4B — C3 product flow · Tutor session engine.

Implements brief §3 (every session):
  1. Kid inputs homework (text | image->OCR | voice->ASR)
  2. GEETA confirms the OCR text with the kid (human-in-the-loop)
  3. GEETA asks the kid to explain their own approach FIRST
  4. Background working: every step passes the Python verifier (C5);
     only verified steps feed hints. Mismatch -> 3 retries -> "let's check
     this one together" fallback.
  5. Stuck -> escalating hint ladder -> similar worked example -> story
     reframe. NEVER the answer to their problem (C6 guard on every reply).
  6. Silent/disengaged -> re-engagement loop (no-blame reset -> concrete
     anchor -> change strategy -> micro-win)
  7. Session end -> event log -> parent + dev reports (C8)

Privacy (hard): session state is in-memory ONLY. Nothing is persisted
beyond the parent/dev report artefacts.

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

from geeta.leak_guard import LeakGuard, GuardLog
from geeta.model_client import ModelClient, MockModel, get_model, load_config
from geeta.tutor_data import TUTOR_SYSTEM, WORK_SYSTEM, ProblemStore
from geeta.verifier import Verifier, VerifiedPlan, VerdictLog, extract_structured_output
from geeta.worksheet import (WorksheetItem, asked_for_number, find_selection, make_tutor, numbers_named,
                             numbers_in_kid_text, page_summary, split_worksheet,
                             strip_selection)

In [ ]:

HINT_MAX = 3

In [ ]:


@dataclass
class SessionEvent:
    ts: float
    kind: str            # input | ocr | ocr_confirm | approach | hint | similar |
                         # story | reengage | solved | verifier | guard | turn
    payload: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {"t": round(self.ts - self._t0, 3) if hasattr(self, "_t0") else self.ts,
                "kind": self.kind, **self.payload}

In [ ]:


class TutorSession:
    """One tutoring session for one kid (in-memory state only)."""

    def __init__(self, cfg: dict, model: Any, *, kid_name: str = "Kid",
                 problem: dict | None = None, kid_profile: dict | None = None,
                 buddy: str = "GEETA"):
        self.cfg = cfg
        self.buddy = buddy            # the kid's world buddy (Bobo, Zoom...) - GEETA's voice
        self.model = model
        self.kid_name = kid_name
        # portal (2026-10-13): short learning summary from this kid's past
        # sessions - {"text": "...", "watch_for": [mistake cats], "year": 2}.
        # Only derived skills/mistake categories, never past transcripts.
        self.kid_profile = kid_profile or {}
        self._item_stats: dict[str, dict] = {}
        self.problem = problem or {}
        self.events: list[SessionEvent] = []
        self.t0 = time.time()
        self.state = "start"
        self.hint_idx = 0
        self.attempts = 0
        self.solved = False
        self.solved_how: str | None = None      # kid's words that closed it
        self.ocr_confirmed: str | None = None
        self._last_ocr_text: str | None = None
        # session-13 fix: rolling kid<->GEETA transcript so every model call
        # sees the conversation so far (it was context-blind before - each
        # turn built [system, current msg] only, so it forgot the problem,
        # forgot its own hints, and contradicted itself across turns).
        self._history: list[dict] = []
        self._lockin_tried = False   # verified-plan lock-in attempted once per session
        self._history_max = 16   # last ~8 exchanges; 262K ctx so tokens aren't the limit
        # worksheet pages (2026-10-13): a photo of a whole page is split into its
        # questions; the kid picks one and pattern/number-word items are tutored
        # by geeta.worksheet (Python-computed hints, no model free-association).
        self.worksheet: list[WorksheetItem] = []
        self.active_item: WorksheetItem | None = None
        self.item_tutor = None
        self.items_done: list[str] = []
        # focus-helper nudges: at most one per quiet spell, never right after
        # the kid or GEETA spoke (2026-10-13: a kid writing on paper got the
        # whole question re-read at them 5-6 times in a row)
        self._last_nudge = 0.0
        self._nudged_since_kid = False
        self.nudge_cooldown_s = 90.0
        self.nudge_quiet_s = 45.0
        self.inactivity_seconds = float(cfg["tutor"]["reengagement"]["inactivity_seconds"])
        self.max_nudges = int(cfg["tutor"]["reengagement"]["max_nudges_before_reset"])
        self._last_activity = time.time()
        self._nudges = 0
        self._reengaged = False
        self._attention_away_streak = 0
        self.attention_enabled = False
        # guard + verifier wiring
        ans = self.problem.get("answer")
        self.guard: Optional[LeakGuard] = LeakGuard(
            ans,
            reprompt_message=cfg["leak_guard"]["reprompt_message"]
            if "leak_guard" in cfg else "I love that you're keen to find out! But the fun is working it out yourself.",
            block_and_reprompt=bool(cfg.get("leak_guard", {}).get("block_and_reprompt", True)),
            match_spoken_numbers=bool(cfg.get("leak_guard", {}).get("match_spoken_numbers", True)),
            match_reformulations=bool(cfg.get("leak_guard", {}).get("match_reformulations", True)),
        ) if ans not in (None, "") else None
        self.guard_log = GuardLog(Path(cfg["paths"]["outputs"]) / "m5" / "leak_triggers.jsonl") \
            if cfg.get("leak_guard", {}).get("log_triggers") else None
        self.verifier = Verifier(expected_answer=ans)
        self.verdict_log = VerdictLog(Path(cfg["paths"]["outputs"]) / "m5" / "verdicts.jsonl") \
            if cfg.get("verifier", {}).get("log_verdicts") else None
        self.max_retries = int(cfg.get("verifier", {}).get("max_regeneration_retries", 3))
        self.fallback_message = cfg.get("verifier", {}).get(
            "fallback_message", "Let's check this one together — can you show me how you'd start it?")

    # -- event bookkeeping -------------------------------------------------
    def _ev(self, kind: str, **payload) -> SessionEvent:
        e = SessionEvent(time.time(), kind, payload)
        self.events.append(e)
        return e

    def _emit_guarded(self, text: str, child_text: str = "") -> str:
        """Send tutor text through the C6 guardrail; log triggers."""
        if self.guard is None:
            return text
        res = self.guard.check(text, child_current_text=child_text)
        if self.guard_log:
            self.guard_log.record(res, {"session": self.kid_name,
                                        "problem": self.problem.get("id")})
        if res.blocked:
            self._ev("guard", trigger=res.leak_found, action=res.action)
        return res.outgoing_text

    # -- input routing -------------------------------------------------------
    @staticmethod
    def _looks_like_maths(text: str) -> bool:
        """A doodle for fun is not a maths question - don't force the
        'here's what I read, is it right?' flow on a drawing of a cat."""
        t = (text or "").strip().lower()
        if len(t) < 3:
            return False
        if any(ch.isdigit() for ch in t):
            return True
        kw = ("plus", "minus", "add", "subtract", "times", "multiply", "divide",
              "share", "how many", "how much", "total", "sum", "equal", "count",
              "shape", "fraction", "half", "quarter", "o'clock", "clock", "cents",
              "dollar", "number", "pattern", "next", "which", "=", "+", "-", "x")
        return any(k in t for k in kw)

    def _image_has_question(self, image_path: str | Path, ocr_text: str) -> bool:
        """Two-stage doodle check: cheap keyword heuristic, then a 1-word vision
        confirm - the OCR model sometimes hallucinates notation (an ellipse ->
        "\\sqrt{0}") so a lone digit in the transcription isn't enough."""
        if not self._looks_like_maths(ocr_text):
            return False
        if not isinstance(self.model, ModelClient):
            return True
        try:
            ans = self.model.chat_image(
                image_path,
                "Look at this image. Does it show a real maths homework question that a "
                "child is meant to answer (numbers, an equation, a word problem, a shape "
                "task)? If it is just a doodle, scribble or picture for fun, say NO. "
                "Reply with exactly one word: YES or NO.",
                temperature=0.0, top_p=0.8, top_k=10, repeat_penalty=1.0, max_tokens=4)
            return "no" not in ans.strip().lower()[:4]
        except Exception:
            return True   # on any error, don't block a possibly-real question

    def feed_image(self, image_path: str | Path) -> str:
        """Canvas/photo input -> OCR -> kid-confirm step (brief §3.2).
        If it's clearly not a maths question (a fun drawing), say so warmly
        instead of pretending to have transcribed a question."""
        if isinstance(self.model, ModelClient):
            text = self.model.ocr_page(image_path)
        else:
            text = self.model.ocr(image_path)
        self._ev("ocr", raw=text, source=str(image_path))

        if not self._image_has_question(image_path, text):
            self._ev("doodle", raw=text[:200])
            self.state = "start"
            return self._emit_guarded(
                "Love that drawing! 🎨 It's not a maths question though - "
                "when you've got one, draw it or snap a photo and I'll help you work it out.")

        self._last_ocr_text = text
        self.state = "ocr_confirm"
        self.worksheet = split_worksheet(text)
        if len(self.worksheet) >= 2:
            lines = [f"{i.label}) {i.text}" for i in self.worksheet if i.kind != "words"]
            words = [i.text for i in self.worksheet if i.kind == "words"]
            if words:
                lines.append("Write in numbers: " + "; ".join(words))
            prompt = ("Here's what I read from your page — is it right?\n"
                      + "\n".join(lines)
                      + "\nSay yes, or tell me if any number is different.")
            return self._emit_guarded(prompt, "")
        prompt = ("Here's what I read from your page — is it right? "
                  f"{text!s}. Tell me if any number or word is different.")
        return self._emit_guarded(prompt, "")

    def confirm_ocr(self, kid_text: str) -> str:
        """Kid confirms or corrects the transcription."""
        low = kid_text.lower()
        accepted = any(w in low for w in ("yes", "right", "correct", "yep", "that's right", "ok"))
        # Session 13 fix: this used to be `self.ocr_confirmed or low`, which is
        # ALWAYS `low` (ocr_confirmed starts None) - so confirming with "yes"
        # silently overwrote the real OCR'd question with the literal word
        # "yes". Use the actual OCR text on accept; the kid's own correction
        # on reject (unchanged, that branch was already right).
        self.ocr_confirmed = (self._last_ocr_text or low) if accepted else kid_text.strip()
        self.state = "approach_first"
        self._ev("ocr_confirm", kid_said=kid_text[:120], accepted=accepted)
        if not accepted:
            fixed = split_worksheet(kid_text)
            if fixed:
                self.worksheet = fixed
        if len(self.worksheet) >= 2:
            # never lock a whole page in as ONE question - that is what made
            # "help with question 4" come back as "next term in pattern 1"
            self.state = "pick"
            pick = find_selection(kid_text, self.worksheet)
            return self._activate_item(pick) if pick else self._emit_guarded(
                ("Great! " if accepted else "Thanks for fixing that! ")
                + page_summary(self.worksheet))
        if len(self.worksheet) == 1 and make_tutor(self.worksheet[0], self.buddy) is not None:
            return self._activate_item(self.worksheet[0])
        self._lock_in_verified_answer()
        if accepted:
            return self._emit_guarded(
                "Great, that's our question locked in. Now — how would YOU start? "
                "Tell me your first little idea, even a wobbly one.")
        return self._emit_guarded(
            "Okay, I've got your version — thanks for fixing that! "
            "Now how would you start?")

    def _lock_in_verified_answer(self) -> None:
        """Session 13 fix (found live-testing): a bank demo problem
        (problem_id) already has a ground-truth `answer`, but REAL usage -
        a kid photographing their own homework - never did. That silently
        broke two things for the entire real-world flow: (1) `solved`
        detection in feed_text() only fires when `self.problem.get("answer")`
        is set, so a real session could never reach "solved" and the parent
        report button never enabled; (2) the never-reveal LeakGuard is only
        constructed in __init__ from that same answer, so it was permanently
        None - completely disabled - for every real (non-bank) session, not
        just occasionally bypassed. Fix: once the kid confirms/corrects the
        OCR'd question, independently compute + verify GEETA's own answer to
        it (same C5 verifier used by the M5 eval harness - not just trusting
        the model), and use that as this session's ground truth going
        forward, activating the leak guard too. No-op if this is already a
        bank problem with a real answer, or if verification doesn't succeed
        (kid just proceeds without solved-detection/leak-guard, as before -
        never a regression, only a strict improvement when it works).
        """
        if self.problem.get("answer") or not self.ocr_confirmed:
            return
        self.problem = dict(self.problem or {})
        self.problem["question_text"] = self.ocr_confirmed
        result = self.verify_background_working(self.ocr_confirmed)
        if not (result["ok"] and result["plan"]):
            return
        answer = result["plan"].get("final_answer")
        if answer in (None, ""):
            return
        # "40c" / "40 cents" / "$0.40" from the model -> 40, so a kid saying "40" counts
        import re as _re
        m = _re.fullmatch(r"\s*(\d+)\s*(?:c|cents?)\s*", str(answer))
        if m:
            answer = int(m.group(1))
        self.problem["answer"] = answer
        # keep the verified WORKING too - the hints the kid sees are grounded
        # in these steps instead of free-associating (see _chat_context).
        steps = result["plan"].get("steps")
        if isinstance(steps, list) and steps:
            self.problem["solution_steps"] = steps
        self.verifier = Verifier(expected_answer=answer)
        if self.guard is None:
            lg_cfg = self.cfg.get("leak_guard", {})
            self.guard = LeakGuard(
                answer,
                reprompt_message=lg_cfg.get(
                    "reprompt_message",
                    "I love that you're keen to find out! But the fun is working it out yourself."),
                block_and_reprompt=bool(lg_cfg.get("block_and_reprompt", True)),
                match_spoken_numbers=bool(lg_cfg.get("match_spoken_numbers", True)),
                match_reformulations=bool(lg_cfg.get("match_reformulations", True)))

    def feed_text(self, kid_text: str) -> str:
        """Typed/spoken text input -> route, then record the turn so the next
        model call has the full conversation."""
        reply = self._route_text(kid_text)
        t = kid_text.strip()
        if t:
            self._history.append({"role": "user", "content": t})
            self._history.append({"role": "assistant", "content": reply})
            if len(self._history) > self._history_max:
                self._history = self._history[-self._history_max:]
        return reply

    def _route_text(self, kid_text: str) -> str:
        """Typed/spoken text input. Routes by session state."""
        self._last_activity = time.time()
        self._nudged_since_kid = False
        t = kid_text.strip()
        from geeta.arith import is_personal_info
        # privacy: never keep phone numbers / addresses a kid types, even in memory
        self._ev("input", kid_text="[private info removed]" if is_personal_info(t) else t[:300])

        if self.state == "ocr_confirm":
            return self.confirm_ocr(t)

        # chit-chat ("what's your name?", "tell me a joke") - friendly, pre-written,
        # then straight back to the question we were on
        from geeta.arith import small_talk, parse_question
        chat = small_talk(t, self.buddy, self.kid_name)
        if chat:
            back = self._tutor_ask()
            return self._emit_guarded(chat + (f" Back to our question: {back}" if back else ""))

        # a typed question Python can read exactly (patterns, sums, word
        # problems...) gets an exact tutor - no model-written hints at all
        busy = self.item_tutor is not None and not self.item_tutor.done
        only_typed = not self.worksheet or all(i.label == "typed" for i in self.worksheet)
        if only_typed and (not busy or t.rstrip().endswith("?")):
            typed = [i for i in split_worksheet(t) if i.kind == "pattern"]
            if len(typed) == 1:
                self.worksheet = typed
                return self._activate_item(typed[0])
            p = parse_question(t)
            if p is not None and (not busy or p.text != (self.active_item.text if self.active_item else "")):
                item = WorksheetItem("typed", "arith", p.text,
                                     answer=p.answer if isinstance(p.answer, int) else None,
                                     meta={"problem": p})
                self.worksheet = [item]
                self.active_item = None
                return self._activate_item(item)

        if self.worksheet:
            pick = find_selection(t, self.worksheet)
            between = self.state == "pick" or (self.item_tutor is not None and self.item_tutor.done)
            if pick is None and between:
                # "22" / "check 21 22 23 24": the first named question not done yet
                named = [i for i in numbers_named(t, self.worksheet) if i.label not in self.items_done]
                pick = named[0] if named else None
            if pick is not None:
                # "word 2 - 375": an answer said in the same breath as the pick
                rest = strip_selection(t, pick.label)
                # "check 21 22 23 24": a list of question numbers - none of them is an answer
                said_answer = bool(numbers_in_kid_text(rest)) and len(numbers_named(t, self.worksheet)) <= 1
                same = pick is self.active_item and self.item_tutor is not None \
                    and not self.item_tutor.done
                if same:
                    return self._item_turn(rest) if said_answer else \
                        self._emit_guarded(self.item_tutor.restate())
                if pick is not self.active_item or said_answer:
                    intro = self._activate_item(pick)
                    if said_answer and self.item_tutor is not None:
                        head = (f"Question {pick.label}" if pick.kind == "pattern"
                                else f"\"{pick.text}\"")
                        return f"{head}: " + self._item_turn(rest)
                    return intro
            n = asked_for_number(t)
            if pick is None and n is not None:
                labels = [i.label for i in self.worksheet if not i.label.startswith("word")]
                if labels and str(n) not in labels:
                    return self._emit_guarded(
                        f"Hmm, I can only see questions {labels[0]} to {labels[-1]} on your page. "
                        + self._whats_left())
            if self.item_tutor is not None and not self.item_tutor.done:
                return self._item_turn(t)
            if self.state == "pick" or (self.item_tutor is not None and self.item_tutor.done):
                return self._emit_guarded(self._whats_left())

        # A TYPED question deserves the same treatment as a photographed one:
        # compute + verify a plan once, so this session also gets grounded
        # hints, the step ladder, the leak guard and solved-detection. Before
        # this, all of that was OCR-path-only and typed problems got none of it.
        if (not self._lockin_tried and not self.problem.get("answer")
                and self._looks_like_maths(t) and len(t) >= 5):
            self._lockin_tried = True
            self.ocr_confirmed = self.ocr_confirmed or t
            try:
                self._lock_in_verified_answer()
            except Exception:
                pass   # never let plan-building break the reply path
            # this message IS the question - don't judge it as an attempt
            self.state = "engaged"
            return self._emit_guarded(
                "Ooh, let's work this one out together! 😊 What's the first thing you notice - "
                "or what would you try first?")

        # did the kid state the final answer?
        if self.problem.get("answer") and self._kid_stated_answer(t):
            if not self.solved:
                self.solved = True
                self.solved_how = t
                self._ev("solved", kid_said=t[:200],
                         after_hints=self.hint_idx, attempts=self.attempts)
            return self._emit_guarded(
                "YES! That's exactly it — and YOU got there. "
                "Tell me: what was the trickiest little part? "
                "I'm putting that in your report. 🌟")

        low = t.lower()
        if self.state == "start" or self.state == "approach_first":
            self.state = "engaged"
            self._ev("approach", kid_text=t[:200])
        if any(w in low for w in ("stuck", "don't know", "idk", "help", "no clue")):
            return self._hint_step(t)
        if self._is_silence(t):
            return self._reengage(t)
        if self._looks_like_progress(t):
            return self._coach_progress(t)
        if self.model is not None:
            reply = self._model_reply(t)
        else:
            reply = "Tell me what you tried!"
        return self._emit_guarded(reply, child_text=t)

    # -- worksheet items (geeta.worksheet) -----------------------------------
    def _activate_item(self, item: WorksheetItem) -> str:
        """Switch the session onto ONE question from the page. Everything that
        was keyed to the previous question (answer, guard, hints, transcript)
        is reset so nothing from question 1 leaks into question 4."""
        self.active_item = item
        self.item_tutor = make_tutor(item, self.buddy)
        self.state = "engaged"
        self.hint_idx = 0
        self._history = []
        self.guard = None
        self.problem = {"question_text": item.text,
                        "id": f"page-q{item.label.replace(' ', '')}"}
        self.ocr_confirmed = item.text
        from geeta.worksheet import skill_label
        self._ev("select", item=item.label, item_kind=item.kind, text=item.text[:200],
                 skill=skill_label(item))
        # typed questions all share the label "typed" - give each its own stats row
        stats_key = item.label if item.label != "typed" else f"typed-{len(self._item_stats) + 1}"
        self._stats_key = stats_key
        self._item_stats.setdefault(stats_key, {
            "label": item.label, "kind": item.kind, "skill": skill_label(item),
            "step": item.step, "solved": False, "hints": 0, "mistakes": []})
        if item.kind == "pattern" and item.finished_on_paper:
            self.item_tutor = None
            self.state = "pick"
            left = [i.label for i in self.worksheet
                    if i.label not in self.items_done and i.kind == "pattern" and i.gaps]
            return self._emit_guarded(
                f"Question {item.label} is already filled in:\n{item.text}\n"
                f"I checked every jump — it goes {'up' if item.step > 0 else 'down'} by "
                f"{abs(item.step)} each time, so it all fits! ✅"
                + (f" Want to try one with gaps, like question {left[0]}?" if left else ""))
        if self.item_tutor is not None:
            return self._emit_guarded(self.item_tutor.intro() + self._memory_tip(item))
        # not a shape Python can tutor exactly -> normal verified model flow,
        # but grounded on THIS question only
        self.verifier = Verifier(expected_answer=None)
        self._lockin_tried = True
        try:
            self._lock_in_verified_answer()
        except Exception:
            pass
        return self._emit_guarded(
            f"Question {item.label}! Here it is:\n{item.text}\n"
            "How would you start? Tell me your first little idea.")

    def _memory_tip(self, item: WorksheetItem) -> str:
        """One gentle reminder from this kid's past sessions, if it fits."""
        watch = set(self.kid_profile.get("watch_for") or [])
        if item.kind == "pattern" and watch & {"jump_size", "jump_finding"}:
            return " 👀 Tip from last time: check how BIG the jump is before you hop!"
        if item.kind == "pattern" and "direction" in watch:
            return " 👀 Tip from last time: are the numbers going up or down?"
        if item.kind == "words" and "zero_placeholder" in watch and (item.answer or 0) % 100 < 10:
            return " 👀 Tip from last time: watch out for the tens place!"
        return ""

    def _item_turn(self, kid_text: str) -> str:
        tut = self.item_tutor
        before = tut.hint_level
        n_miss = len(tut.mistakes)
        reply = tut.turn(kid_text)
        st = self._item_stats.get(getattr(self, "_stats_key", self.active_item.label))
        for m in tut.mistakes[n_miss:]:
            self._ev("mistake", item=self.active_item.label, **m)
            if st is not None:
                st["mistakes"].append(m["cat"])
        if tut.hint_level > before:
            self.hint_idx += 1
            if st is not None:
                st["hints"] += 1
            self._ev("hint", depth=tut.hint_level, hint=reply[:200], source="worksheet")
        if tut.done and self.active_item.label not in self.items_done and getattr(tut, "unchecked", False):
            # a word puzzle GEETA can't check: it's finished, but not counted as "solved"
            self.items_done.append(self.active_item.label)
            return reply
        if tut.done and self.active_item.label not in self.items_done:
            if st is not None:
                st["solved"] = True
            self.items_done.append(self.active_item.label)
            self.solved = True
            self.solved_how = self.solved_how or kid_text
            self._ev("solved", kid_said=kid_text[:200], item=self.active_item.label,
                     after_hints=self.hint_idx, attempts=self.attempts)
        return reply   # every number in it is Python-computed or the kid's own

    def _whats_left(self) -> str:
        # every question still to do - including ones only the checked model path can help with
        left = [i.label for i in self.worksheet
                if i.label not in self.items_done and not i.finished_on_paper]
        if not left:
            return "You've done every question on this page! 🏆 Amazing work today."
        qs = [l for l in left if not l.startswith("word")]
        ws = [l for l in left if l.startswith("word")]
        bits = []
        if qs:
            bits.append(f"\"question {qs[0]}\"")
        if ws:
            bits.append(f"\"{ws[0]}\" (write a number)")
        return ("Which one next? Still to do: "
                + ", ".join(qs + ws) + ". Just say " + " or ".join(bits) + "!")

    def _coach_progress(self, kid_text: str) -> str:
        """Kid produced a number / attempted result (question Python couldn't
        read exactly). Right/wrong is decided HERE, never by the model."""
        ans = self.problem.get("answer")
        if ans not in (None, "") and not self._kid_stated_answer(kid_text):
            # we know it's not the answer: say so kindly, then a model nudge (gated)
            lead = "Hmm, not quite - good try though! "
            if isinstance(self.model, MockModel):
                return self._emit_guarded(lead + "Let's look at it again together.", child_text=kid_text)
            msgs = self._chat_context()
            msgs.append({"role": "user", "content":
                         f"The kid answered {kid_text}, which is NOT correct. Do not say it is right. "
                         "Give ONE small, kind clue for the next step, using only numbers from the question."})
            reply = self._gate_reply(lambda: self.model.chat(msgs, temperature=0.4, top_p=0.9, max_tokens=90),
                                     kid_text, wrong_attempt=True)
            return self._emit_guarded(lead + reply, child_text=kid_text)
        if isinstance(self.model, MockModel):
            reply = self.model.respond(kid_text, child_visible=self.ocr_confirmed or "")
            return self._emit_guarded(reply, child_text=kid_text)
        msgs = self._chat_context()
        msgs.append({"role": "user",
                     "content": f"The kid says: {kid_text} — coach the next little step."})
        reply = self._gate_reply(lambda: self.model.chat(msgs, temperature=0.4, top_p=0.9, max_tokens=90),
                                 kid_text, wrong_attempt=ans not in (None, ""))
        return self._emit_guarded(reply, child_text=kid_text)

    # -- hallucination gate for anything the language model writes ------------
    SAFE_FALLBACK = ("Let's work it out together! Tell me the very first step you'd try - "
                     "or a number you get along the way - and I'll help from there. 😊")

    def _allowed_numbers(self, kid_text: str) -> set:
        import re as _re
        q = self.problem.get("question_text") or self.ocr_confirmed or ""
        allowed = set(range(0, 13))
        for src in [q, kid_text] + [m.get("content", "") for m in self._history if m.get("role") == "user"]:
            allowed |= {float(x) for x in _re.findall(r"(?<![\d.])\d+(?:\.\d+)?", src or "")}
        for st in self.problem.get("solution_steps") or []:
            if isinstance(st, dict):
                for v in (st.get("result"),):
                    try:
                        allowed.add(float(v))
                    except (TypeError, ValueError):
                        pass
        return {float(a) for a in allowed}

    def _reply_problems(self, reply: str, kid_text: str, wrong_attempt: bool) -> list[str]:
        import re as _re
        probs = []
        ops = {"+": lambda a, b: a + b, "plus": lambda a, b: a + b, "-": lambda a, b: a - b,
               "−": lambda a, b: a - b, "minus": lambda a, b: a - b, "take away": lambda a, b: a - b,
               "x": lambda a, b: a * b, "×": lambda a, b: a * b, "*": lambda a, b: a * b,
               "times": lambda a, b: a * b, "÷": lambda a, b: a / b if b else None,
               "/": lambda a, b: a / b if b else None, "divided by": lambda a, b: a / b if b else None}
        for a, op, b, c in _re.findall(r"(\d+(?:\.\d+)?)\s*(\+|plus|-|−|minus|take away|x|×|\*|times|÷|/|divided by)\s*"
                                       r"(\d+(?:\.\d+)?)\s*(?:=|is|makes|equals|gives|would be)\s*(?:only\s+)?(\d+(?:\.\d+)?)", reply, _re.I):
            v = ops[op.lower()](float(a), float(b))
            if v is None or abs(v - float(c)) > 1e-9:
                probs.append(f"false sum {a}{op}{b}={c}")
        if _re.search(r"\b(more|less) than \d+ but (less|more) than \d+", reply, _re.I):
            probs.append("range claim")                     # "more than 14 but less than 20" - unverifiable
        if wrong_attempt and _re.search(r"\b(yes|yay|yep|correct|that'?s (right|correct|it)|exactly|well done|"
                                        r"great job|perfect|you got it|spot on|brilliant|awesome|see, you|nailed|"
                                        r"you did it|you did that|you found it|you worked (it|that) out)\b", reply, _re.I):
            probs.append("praises a wrong answer")
        allowed = self._allowed_numbers(kid_text)
        extra = [x for x in _re.findall(r"(?<![\d.])\d+(?:\.\d+)?", reply) if float(x) not in allowed]
        if extra:
            probs.append("unsupported numbers " + ",".join(extra[:4]))
        if _re.search(r"dollars and cents|first number on your fingers|left to right", reply, _re.I) and \
                "$" not in (self.problem.get("question_text") or ""):
            probs.append("off-topic stock phrase")
        return probs

    def _gate_reply(self, gen, kid_text: str, wrong_attempt: bool = False) -> str:
        """Ask the model up to twice; only a reply with no detectable problem
        reaches the kid. Otherwise a safe, honest fallback. Logged for review."""
        last = []
        for attempt in range(2):
            try:
                reply = (gen() or "").strip()
            except Exception:
                reply = ""
            if not reply:
                continue
            last = self._reply_problems(reply, kid_text, wrong_attempt)
            if not last:
                return reply
            self._ev("gate_blocked", reply=reply[:200], problems=last)
        self._ev("gate_fallback")
        return self.SAFE_FALLBACK

    def _tutor_ask(self) -> str:
        """The question the active exact tutor is waiting on (for chit-chat returns)."""
        tut = self.item_tutor
        if tut is None or tut.done:
            return ""
        for name in ("_ask", "_ask_gap"):
            f = getattr(tut, name, None)
            if callable(f):
                try:
                    if name == "_ask_gap" and not getattr(tut, "jump_known", True):
                        return tut._ask_jump()
                    return f()
                except Exception:
                    pass
        return f"What's \"{getattr(tut.item, 'text', '')}\" in numbers?"

    # -- hint ladder (C3 §3.5) ----------------------------------------------
    def _hint_step(self, kid_text: str) -> str:
        self.attempts += 1
        if self._reengaged:
            self._reengaged = False
        p = self.problem
        if self.hint_idx < HINT_MAX and p.get("hint_ladder"):
            h = p["hint_ladder"][min(self.hint_idx, len(p["hint_ladder"]) - 1)]
            self.hint_idx += 1
            self._ev("hint", depth=self.hint_idx, hint=h[:200])
            text = f"Here's a nudge: {h} Try that and tell me what happens."
            out = self._emit_guarded(text, child_text=kid_text)
            if self.guard is not None and self.guard.check(text, child_current_text=kid_text).blocked:
                # conservative over-block on a hint that contains the answer value;
                # swap to a number-free nudge so the kid still gets a step
                return self._emit_guarded(
                    "Let's go slower. Read the question with me, then tell me the very first "
                    "little thing you'd work out. 😊")
            return out
        # no authored ladder, but we have verified working -> build the hint
        # from the next unrevealed step instead of the generic catch-all.
        steps = p.get("solution_steps")
        if self.hint_idx < HINT_MAX and steps:
            idx = min(self.hint_idx, len(steps) - 1)
            s = steps[idx] if isinstance(steps[idx], dict) else {}
            why = (s.get("explanation") or "").strip()
            expr = s.get("expression")
            self.hint_idx += 1
            if why:
                nudge = f"Here's a nudge: {why}. Try just that bit and tell me what you get."
            elif expr:
                nudge = f"Here's a nudge: start with {expr}. What do you get?"
            else:
                nudge = None
            if nudge and self._reply_problems(nudge, kid_text, False):
                nudge = None          # the model-written step text failed the hallucination gate
            if nudge:
                self._ev("hint", depth=self.hint_idx, hint=nudge[:200], source="verified_steps")
                if self.guard is not None and self.guard.check(nudge, child_current_text=kid_text).blocked:
                    return self._emit_guarded(
                        "Let's go slower. Read the question with me, then tell me the very first "
                        "little thing you'd work out. 😊")
                return self._emit_guarded(nudge, child_text=kid_text)

        sw = p.get("similar_worked")
        if sw:
            self._ev("similar", text=sw[:200])
            return self._emit_guarded(
                f"Let's look at a SIMILAR one first: {sw} "
                "Now back to YOURS — what's the first little step?",
                child_text=kid_text)
        story = p.get("story_reframe")
        if story:
            self._ev("story", text=story[:200])
            return self._emit_guarded(f"{story} Does that help you start?",
                                       child_text=kid_text)
        return self._emit_guarded(
            "Let's slow right down. Break it into the smallest piece and do just that one.",
            child_text=kid_text)

    # -- on-device attention signal (session 13, user-approved + consented;
    # brief C10 was Phase-2/HREC-gated for camera use in general - this is
    # scoped narrowly per that approval: the browser's on-device face model
    # NEVER sends a frame/video anywhere, only a short state string
    # ("engaged"/"away") reaches this method. Nothing here stores or forwards
    # imagery; only the derived text state is logged as a normal
    # SessionEvent, same as any other event, for the live re-engagement loop
    # and the parent report's focus summary.) ---------------------------------
    def attention_signal(self, state: str) -> str | None:
        """Feed a privacy-safe engagement label ("engaged"/"away") from the
        on-device camera detector. Returns a reply ONLY when sustained
        distraction triggers a re-engagement nudge; None otherwise (caller
        should not add a chat bubble for a no-op signal)."""
        self._ev("attention", state=state)
        if state != "away":
            self._attention_away_streak = 0
            return None
        self._attention_away_streak += 1
        if self._attention_away_streak < 3:   # ~a few consecutive samples, not one blip
            return None
        self._attention_away_streak = 0
        now = time.time()
        if (self._nudged_since_kid                                # one nudge, then wait for the kid
                or now - self._last_activity < self.nudge_quiet_s   # they just spoke / are working
                or now - self._last_nudge < self.nudge_cooldown_s):
            return None
        if self.worksheet and (self.item_tutor is None or self.item_tutor.done):
            return None   # between questions: nothing to pull them back to
        self._last_nudge = now
        self._nudged_since_kid = True
        return self._reengage("", trigger="attention")

    # -- re-engagement loop (brief §3.6) -------------------------------------
    def _reengage(self, kid_text: str, trigger: str = "inactivity") -> str:
        if self.item_tutor is not None and not self.item_tutor.done:
            # stay on the kid's actual question - not a generic "10 fingers" anchor
            self._ev("reengage", nudge=self._nudges, trigger=trigger, source="worksheet")
            return self._emit_guarded(self.item_tutor.nudge())
        if self._nudges < self.max_nudges:
            self._nudges += 1
            self._ev("reengage", nudge=self._nudges, trigger=trigger)
            if self._nudges == 1:
                msg = ("No worries at all. Let's just start with the very first little bit."
                       if trigger != "attention" else
                       "Hey, welcome back! No worries at all - let's pick up right here.")
                return self._emit_guarded(msg, child_text=kid_text)
            return self._emit_guarded(
                "Here's a trick that always helps: hold up 10 fingers. "
                "Can you show me the first number on your fingers?",
                child_text=kid_text)
        # reset: no-blame + concrete anchor + change strategy + micro-win
        self._nudges = 0
        self._reengaged = True
        self._ev("reengage", reset=True, trigger=trigger)
        p = self.problem
        anchor = "Try it on your fingers first — show me what your fingers say."
        story = p.get("story_reframe")
        return self._emit_guarded(
            f"Fresh start, no worries. {story + ' ' if story else ''}{anchor} "
            "Whatever you get, I'm right here with you. 💪",
            child_text=kid_text)

    # -- verified background working (C5) ------------------------------------
    def verify_background_working(self, prompt: str | None = None) -> dict:
        """Model works the problem privately; every step verified.

        Returns {"ok": bool, "plan": dict|None, "verdicts": [...],
                 "fallback_used": bool, "final": str}
        """
        from geeta.verifier import VerifiedPlan, try_deterministic_solve
        vp = VerifiedPlan(self.verifier, max_retries=self.max_retries,
                          fallback_message=self.fallback_message)

        # Python proposes first when it can be EXACT (money token sums, bare
        # arithmetic). This is the same "model proposes, Python disposes"
        # contract - just with Python allowed to propose where it is certain.
        # Fixes the money shape, where the LoRA emitted bracket-list garbage
        # instead of JSON so no verified plan existed at all (which also left
        # leak-guard + solved-detection disabled for those sessions).
        det = try_deterministic_solve(self.problem.get("question_text", "")
                                      or self.ocr_confirmed or "")
        if det is not None:
            v = self.verifier.verify(det)
            if v.ok:
                if self.verdict_log:
                    self.verdict_log.record(v, meta={"session": self.kid_name,
                                                     "problem": self.problem.get("id"),
                                                     "source": "deterministic"})
                self._ev("verifier", ok=True, attempts=0, fallback=False, source="deterministic")
                return {"ok": True, "plan": det, "verdicts": [v.to_dict()],
                        "fallback_used": False, "final": "ok"}
        if isinstance(self.model, MockModel):
            gen = lambda p: self.model.background_working(kid_visible=p or "")
        else:
            # Session 13 fix: same formatting rules as 05_e2e_text_loop.ipynb's
            # WORKING_PROMPT_TMPL - keeps the live webapp path and the offline
            # eval harness consistent (both were losing solved problems to
            # trivial answer-format mismatches, not wrong arithmetic).
            base_prompt = (
                "Work this problem step by step. Emit ONLY JSON: "
                '{"steps": [{"step": 1, "expression": "...", "result": ...,'
                ' "explanation": "..."}], "final_answer": ...}. '
                "Formatting rules for final_answer: a clock time is H:MM or "
                'HH:MM (e.g. "11:00", not 11). A number spelled out in words '
                'uses spaces, not hyphens (e.g. "ninety eight", not '
                '"ninety-eight"). A step whose value is not a plain '
                "arithmetic expression (a word, a shape name, counting "
                "reasoning) may have expression: null - only give an "
                "expression when it is literally re-computable arithmetic. "
                f'Problem: {self.problem.get("question_text", "")}')
            # R1 (session 12): use WORK_SYSTEM, not TUTOR_SYSTEM, for this
            # internal-only call - the kid-facing never-reveal persona made
            # the model treat its own working request as a leak probe and
            # frequently refuse to emit the JSON at all. Nothing from this
            # call reaches the child directly; it is checked by the verifier
            # first (matches PROJECT_BRIEF Known risk R2).
            # Also thread the verifier's corrective retry note (VerifiedPlan.run
            # passes it as ``p`` on retries) back into the prompt instead of
            # silently discarding it and re-sending the same original prompt.
            def gen(p):
                wp = base_prompt
                if p and p not in (None, prompt, "work it"):
                    wp = base_prompt + "\n\n" + p
                return self.model.chat(
                    [{"role": "system", "content": WORK_SYSTEM},
                     {"role": "user", "content": wp}],
                    temperature=0.2, top_p=0.9, repeat_penalty=1.1, max_tokens=384)
        # session-13: capture every raw model attempt so a fallback can be
        # reviewed and turned into training data later (outputs/m5/fallback_cases.jsonl)
        _raw_attempts = []
        _gen0 = gen
        def gen(p, _g=_gen0):
            out = _g(p)
            _raw_attempts.append(out if isinstance(out, str) else json.dumps(out, default=str))
            return out

        plan, verdicts, fallback = vp.run(gen, prompt or "work it")
        for v in verdicts:
            if self.verdict_log:
                self.verdict_log.record(_v_from_dict(v),
                                        meta={"session": self.kid_name,
                                              "problem": self.problem.get("id")})
        if fallback or plan is None:
            try:
                import time as _t
                fc = Path(self.cfg["paths"]["outputs"]) / "m5" / "fallback_cases.jsonl"
                fc.parent.mkdir(parents=True, exist_ok=True)
                with fc.open("a") as _f:
                    _f.write(json.dumps({
                        "ts": _t.strftime("%Y-%m-%dT%H:%M:%S%z"),
                        "question": self.problem.get("question_text") or self.ocr_confirmed or "",
                        "problem_id": self.problem.get("id"),
                        "skill_id": self.problem.get("skill_id"),
                        "expected_answer": self.problem.get("answer"),
                        "n_attempts": len(_raw_attempts),
                        "raw_attempts": _raw_attempts,
                        "verdict_reasons": [v.get("reason") for v in verdicts],
                    }, default=str) + "\n")
            except Exception:
                pass
        self._ev("verifier", ok=self.verifier_ok(plan, verdicts),
                 attempts=len(verdicts), fallback=fallback)
        final = "ok" if plan else self.fallback_message
        return {"ok": plan is not None and self.verifier_ok(plan, verdicts),
                "plan": plan, "verdicts": verdicts, "fallback_used": fallback,
                "final": final}

    def verifier_ok(self, plan: dict | None, verdicts: list[dict]) -> bool:
        if plan is None:
            return False
        return all(s.get("ok", False) for s in verdicts[-1].get("steps", []))

    # -- model reply ---------------------------------------------------------
    def _chat_context(self) -> list[dict]:
        """System prompt + pinned problem + rolling transcript. Every model
        call in the tutoring loop goes through this so nothing is context-blind."""
        msgs = [{"role": "system", "content": TUTOR_SYSTEM}]
        if self.kid_profile.get("text"):
            msgs.append({"role": "user", "content":
                "PRIVATE background about this child from earlier sessions (never read "
                "it out; use it to pitch hints at their level and praise real progress):\n"
                + self.kid_profile["text"]})
            msgs.append({"role": "assistant", "content": "Understood."})
        q = self.problem.get("question_text") or self.ocr_confirmed
        if q:
            msgs.append({"role": "user", "content": f"The question we are working on: {q}"})
            msgs.append({"role": "assistant", "content": "Got it. Let's work it out together."})
        # Ground the tutoring in the VERIFIED working. Without this the model
        # free-associates (it once answered a "share 12 between 4" question
        # with "put them in bags of ten") because the plan we had already
        # computed and verified was never shown to it.
        steps = self.problem.get("solution_steps")
        if steps:
            lines = []
            for s in steps[:6]:
                if not isinstance(s, dict):
                    continue
                expr, res = s.get("expression"), s.get("result")
                why = (s.get("explanation") or "").strip()
                if expr and res is not None:
                    lines.append(f"- {expr} = {res}" + (f"  ({why})" if why else ""))
                elif why:
                    lines.append("- " + why)
            if lines:
                msgs.append({"role": "user", "content":
                    "PRIVATE - the child must NEVER see this. The checked working is:\n"
                    + "\n".join(lines)
                    + "\nUse it to pick the ONE next small step to nudge them toward. "
                      "Never state the final answer or read these lines out."})
                msgs.append({"role": "assistant",
                             "content": "Understood - I'll guide the next step only."})
        msgs.extend(self._history[-self._history_max:])
        return msgs

    def _model_reply(self, kid_text: str) -> str:
        if isinstance(self.model, MockModel):
            return self.model.respond(kid_text, child_visible=self.ocr_confirmed or "")
        msgs = self._chat_context()
        msgs.append({"role": "user", "content": kid_text})
        return self._gate_reply(lambda: self.model.chat(msgs, temperature=0.4, top_p=0.9, max_tokens=100),
                                kid_text)

    # -- helpers -------------------------------------------------------------
    def _kid_stated_answer(self, kid_text: str) -> bool:
        if not self.guard:
            return False
        from geeta.leak_guard import answer_in_text
        return answer_in_text(self.guard.kind, self.guard.val,
                              self.guard.raw_answer, kid_text)

    @staticmethod
    def _is_silence(t: str) -> bool:
        low = t.lower()
        return low in ("", "...", "…", "hmm", "uh", "shh", "zzz") or low.startswith(("...", "…"))

    @staticmethod
    def _looks_like_progress(t: str) -> bool:
        from geeta.leak_guard import _NUM_TOK
        return bool(_NUM_TOK.search(t))

    def close(self) -> dict:
        """Session summary (feeds C8 reports)."""
        dur = time.time() - self.t0
        hints = [e for e in self.events if e.kind == "hint"]
        reeng = [e for e in self.events if e.kind == "reengage"]
        attn = [e for e in self.events if e.kind == "attention"]
        attn_reeng = [e for e in reeng if e.payload.get("trigger") == "attention"]
        focus = None
        if attn:
            engaged_n = sum(1 for e in attn if e.payload.get("state") == "engaged")
            focus = {
                "samples": len(attn),
                "engaged_pct": round(100 * engaged_n / len(attn), 1),
                "refocus_moments": len(attn_reeng),
            }
        return {
            "kid": self.kid_name,
            "problem_id": self.problem.get("id"),
            "skill_id": self.problem.get("skill_id"),
            "year": self.problem.get("year") or self.kid_profile.get("year"),
            "strand": self.problem.get("strand"),
            "solved": self.solved,
            "solved_how": self.solved_how,
            "hints_used": len(hints),
            "max_hint_depth": self.hint_idx,
            "reengagements": len(reeng),
            "attempts": self.attempts,
            "duration_s": round(dur, 1),
            "ocr_confirmed": self.ocr_confirmed,
            "focus": focus,   # None unless the kid opted in to the on-device attention helper
            "n_events": len(self.events),
            "items": list(self._item_stats.values()),   # per worksheet question
            "events": [e.to_dict() for e in self.events],
        }

In [ ]:


def _v_from_dict(d: dict):
    from geeta.verifier import Verdict, StepVerdict
    steps = [StepVerdict(s.get("step", i), s.get("expression", "?"),
                         s.get("model_result"), s.get("recomputed"),
                         s.get("ok", False), s.get("reason", ""))
             for i, s in enumerate(d.get("steps", []), 1)]
    return Verdict(d.get("ok", False), d.get("final_model_answer"),
                   d.get("final_expected"), steps, d.get("reason", ""),
                   d.get("elapsed_ms", 0.0))